# Transformer Seq2Seq Baseline — Length Extrapolation Ablation

This notebook trains `TransformerSeq2Seq`, the **global autoregressive baseline**
that is expected to **fail catastrophically** on out-of-distribution sequence lengths.

## Why this model is designed to fail (paper narrative)

| Component | CTC baseline | Transformer (this) |
|---|---|---|
| Encoder | 2D CNN + 1D Dilated CNN (RF≈60) | 2D CNN + Linear projector |
| Positional encoding | **None** | **Absolute sinusoidal PE** ← failure #1 |
| Decoding | **CTC (non-autoregressive)** | **Autoregressive + <EOS>** ← failure #2 |
| Attention scope | **Local (RF≈60 steps)** | **Global self-attention** ← failure #3 |
| Loss | CTCLoss | CrossEntropyLoss |

## Data Pipeline
All dataset and utility code is **reused verbatim from `src/ctc/`** to
guarantee a fair, apples-to-apples comparison. The only difference is the model.

## Expected outcome
- `L ≤ 12` (in-distribution): ~60–80% sequence accuracy ✅
- `L > 12` (OOD): catastrophic collapse to <10% by L=50 ❌

This demonstrates that the CTC model's length robustness is not a coincidence
but a structural property of its non-autoregressive, position-agnostic design.

## 1 · Mount Google Drive & Create Checkpoint Directory

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ── Checkpoint directory (separate from CTC checkpoints) ─────────────────────
CHECKPOINT_DIR = "/content/drive/MyDrive/digit-sequence-reader/checkpoint_transformer_baseline"
METRICS_DIR    = os.path.join(CHECKPOINT_DIR, "metrics")
BEST_CKPT      = os.path.join(CHECKPOINT_DIR, "best_transformer.pt")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(METRICS_DIR,    exist_ok=True)

print(f"Checkpoint directory : {CHECKPOINT_DIR}")


## 2 · Clone Repository & Install Dependencies

In [ ]:
REPO_URL = "https://github.com/AmineAitLaamim/digit-sequence-reader.git"
!git clone {REPO_URL}
%cd digit-sequence-reader
!pip install -r requirements.txt -q
!pip install albumentations opencv-python-headless -q

## 3 · Verify GPU

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device       :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4 · Path Setup & Imports

We import the **new** `TransformerSeq2Seq` model but reuse the **existing**
`src/ctc/` dataset and utilities unchanged — this is the key to a fair ablation.

In [ ]:
import sys, os

# Add the repo root so both src.transformer_baseline and src.ctc are importable.
ROOT = os.path.abspath('.')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# ── New Transformer baseline model ──────────────────────────────────────────
from src.transformer_baseline.model import TransformerSeq2Seq

# ── Reusing existing dataset/utils to maintain DRY principles and ensure ────
# ── identical data pipeline for fair comparison. ────────────────────────────
from src.ctc.dataset import (
    InfiniteCTCDataset,
    collate_fn          as ctc_collate_fn,
    get_dataloaders,
    build_multidigit_bank,
    get_digit_aug_pipeline,
    make_sequence,
)
from src.ctc.config import config   # shared config dict

def transformer_collate_fn(batch):
    """Wraps CTC collate to return 2D padded targets for the Transformer."""
    ctc_dict = ctc_collate_fn(batch)
    B = ctc_dict['target_lengths'].size(0)
    max_L = int(ctc_dict['target_lengths'].max().item())
    targets_2d = torch.full((B, max_L), 10, dtype=torch.long)
    cursor = 0
    for i, L in enumerate(ctc_dict['target_lengths'].tolist()):
        targets_2d[i, :L] = ctc_dict['targets'][cursor: cursor + L]
        cursor += L
    ctc_dict['targets'] = targets_2d
    return ctc_dict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW
import math, csv



## 5 · Hyperparameters

> **Fair comparison note.**  
> We use the **exact same** `max_length=12`, `batch_size=64`, `lr=1e-3`,
> and augmentation schedule as the CTC baseline runs.  
> The only addition is `WARMUP_STEPS=2000` which is **mandatory** for
> Transformer stability (not needed for CTC/CNN models).

In [ ]:
# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE   = 64
LR           = 1e-3
WEIGHT_DECAY = 1e-4
EPOCHS       = 30
CLIP_GRAD    = 1.0
NUM_WORKERS  = 2

# ── Data / curriculum settings ───────────────────────────────────────────────
# IMPORTANT: Keep training short (max_length=12) — SAME as CTC baseline.
# This is critical for the fair comparison: both models train on identical data.
TRAIN_MAX_LENGTH = 12    # same cap used for baseline CRNN_CTC
TRAIN_SIZE       = 100_000
VAL_SIZE         = 10_000

# ── Transformer-specific ─────────────────────────────────────────────────────
# WARMUP_STEPS is CRITICAL for Transformer stability.
# Without warmup, Adam starts with huge updates on near-zero gradients in
# the attention layers, causing divergence in the first epoch.
WARMUP_STEPS = 2000

# ── Model architecture ───────────────────────────────────────────────────────
D_MODEL         = 256
NHEAD           = 8
NUM_LAYERS      = 4
DIM_FEEDFORWARD = 1024
DROPOUT         = 0.1

# ── LR scheduler (fallback ReduceLROnPlateau) ────────────────────────────────
LR_PATIENCE = 5
LR_FACTOR   = 0.5
LR_MIN      = 1e-6

# ── Early stopping ───────────────────────────────────────────────────────────
EARLY_STOP_PATIENCE = 12

# ── Override shared config dict (same pattern as CTC notebooks) ──────────────
config['batch_size']        = BATCH_SIZE
config['max_seq_len_final'] = TRAIN_MAX_LENGTH
config['train_size']        = TRAIN_SIZE
config['val_size']          = VAL_SIZE
config['num_workers']       = NUM_WORKERS

print("Hyperparameters set ✔")
print(f"  Batch size        : {BATCH_SIZE}")
print(f"  Learning rate     : {LR}")
print(f"  Warmup steps      : {WARMUP_STEPS}  (CRITICAL for Transformer)")
print(f"  Epochs            : {EPOCHS}")
print(f"  Max train length  : {TRAIN_MAX_LENGTH}  (short — for extrapolation test)")

## 6 · Optional Shape Sanity-Check

In [ ]:
!python -m src.transformer_baseline.model


## 7 · Build Digit Bank & Data Loaders

Identical call to `get_dataloaders()` as in the CTC notebooks — same datasets
(EMNIST + QMNIST + USPS), same augmentation pipeline.

In [ ]:
# Reusing existing get_dataloaders() — identical to the CTC baseline runs.
print("Building digit bank and validation loader (first run downloads datasets)... ")
digit_bank, val_loader, _ = get_dataloaders(data_path=config['data_path'])

# ── CRITICAL FIX: Recreate val_loader with transformer_collate_fn ─────────────
# The default val_loader uses ctc_collate_fn (1D targets). The Transformer 
# requires 2D padded targets for teacher forcing.
val_loader = DataLoader(
    val_loader.dataset,
    batch_size=BATCH_SIZE,
    collate_fn=transformer_collate_fn,  # <--- Use the 2D padding wrapper!
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
)

STEPS_PER_EPOCH = TRAIN_SIZE // BATCH_SIZE
TOTAL_STEPS     = EPOCHS * STEPS_PER_EPOCH
print(f"Steps per epoch : {STEPS_PER_EPOCH} ")
print(f"Total steps     : {TOTAL_STEPS} ")


## 8 · Model Initialisation

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training device : {device}")

# Instantiate the Transformer baseline.
model = TransformerSeq2Seq(
    d_model        = D_MODEL,
    nhead          = NHEAD,
    num_layers     = NUM_LAYERS,
    dim_feedforward= DIM_FEEDFORWARD,
    dropout        = DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters      : {n_params:,}")
print(f"Architecture    : {NUM_LAYERS}-layer TransformerDecoder, d_model={D_MODEL}, "
      f"nhead={NHEAD}, FF={DIM_FEEDFORWARD}")

# ── Optimiser ────────────────────────────────────────────────────────────────
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# ── LR Scheduler: linear warmup → cosine decay ───────────────────────────────
# CRITICAL for Transformer stability — 2000 warmup steps.
# Also set up a ReduceLROnPlateau as a fallback / floor guard.
class WarmupCosineLR:
    """Linear warmup then cosine decay to eta_min."""
    def __init__(self, optimizer, warmup_steps, total_steps, base_lr, eta_min=1e-6):
        self.optimizer    = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps  = total_steps
        self.base_lr      = base_lr
        self.eta_min      = eta_min
        self._step        = 0

    def step(self):
        self._step += 1
        lr = self._get_lr()
        for pg in self.optimizer.param_groups:
            pg['lr'] = lr

    def _get_lr(self):
        s = self._step
        if s <= self.warmup_steps:
            return self.base_lr * s / max(1, self.warmup_steps)
        progress = (s - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        cosine   = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.eta_min + (self.base_lr - self.eta_min) * cosine

    @property
    def current_lr(self):
        return self.optimizer.param_groups[0]['lr']

lr_scheduler = WarmupCosineLR(
    optimizer    = optimizer,
    warmup_steps = WARMUP_STEPS,
    total_steps  = TOTAL_STEPS,
    base_lr      = LR,
    eta_min      = LR_MIN,
)

# ── Optional: resume from existing checkpoint ─────────────────────────────────
start_epoch  = 1
best_val_seq = 0.0

if os.path.exists(BEST_CKPT):
    ckpt = torch.load(BEST_CKPT, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    try:
        optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    except Exception:
        pass
    start_epoch  = ckpt['epoch'] + 1
    best_val_seq = ckpt.get('val_seq_acc', 0.0)
    print(f"Resumed from epoch {ckpt['epoch']} | val_seq_acc={best_val_seq:.4f}")
else:
    print("No existing checkpoint — training from scratch.")

## 9 · Helper Functions

Metric computation uses the same Levenshtein-based CER as the CTC baseline run.
The `validate` function uses **greedy autoregressive decoding** (not teacher-forcing)
so the validation metrics reflect true inference performance.

In [ ]:
# ── Levenshtein distance (CER numerator) ─────────────────────────────────────
def _levenshtein(a, b):
    """Standard DP edit distance."""
    if len(a) < len(b):
        a, b = b, a
    if not b:
        return len(a)
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (ca != cb))
        prev = cur
    return prev[-1]


def compute_metrics(
    decoded: list[list[int]],
    target_lengths: torch.Tensor,
    targets: torch.Tensor, # Can be 1D flat or 2D padded
) -> tuple[float, float]:
    """Compute sequence accuracy and character error rate. Handles both 1D and 2D targets."""
    B = len(decoded)
    seq_correct   = 0
    edit_distance = 0
    total_chars   = 0

    # Check if targets are 1D (flat CTC format) or 2D (padded Transformer format)
    is_1d = (targets.dim() == 1)
    cursor = 0

    for i, length in enumerate(target_lengths.tolist()):
        length = int(length)
        
        if is_1d:
            # 1D flat tensor (from default val_loader)
            gt = targets[cursor:cursor + length].tolist()
            cursor += length
        else:
            # 2D padded tensor (from transformer_collate_fn)
            gt = targets[i, :length].tolist()

        pred = decoded[i]

        if pred == gt:
            seq_correct += 1
        edit_distance += _levenshtein(pred, gt)
        total_chars   += length

    seq_acc = seq_correct / max(1, B)
    cer     = edit_distance / max(1, total_chars)
    return seq_acc, cer


def save_checkpoint(model, optimizer, epoch, val_loss, val_seq_acc, path):
    """Save model + optimizer state to disk."""
    torch.save({
        'epoch':                epoch,
        'model_state_dict':     model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_loss':             val_loss,
        'val_seq_acc':          val_seq_acc,
    }, path)


@torch.no_grad()
def validate(model, dataloader, device):
    """Run a full validation pass using greedy_autoregressive_decode.

    Returns (avg_loss, seq_acc, char_acc).
    """
    from tqdm import tqdm
    model.eval()
    total_loss = total_seq = total_cer = 0.0
    n = 0
    for batch in tqdm(dataloader, desc='  val  ', leave=False):
        images         = batch['images'].to(device)
        targets        = batch['targets'].to(device)
        target_lengths = batch['target_lengths'].to(device)

        # Loss via teacher-forcing.
        loss, _ = model(images, targets=targets, target_lengths=target_lengths)
        total_loss += loss.item()

        # Metrics via true autoregressive decoding.
        max_len      = int(target_lengths.max().item()) * 2 + 5
        enc_feat     = model(images)   # inference mode
        decoded      = model.greedy_autoregressive_decode(enc_feat, max_len=max_len)
        seq_acc, cer = compute_metrics(decoded, target_lengths, targets)
        total_seq   += seq_acc
        total_cer   += cer
        n           += 1

    return total_loss / max(1, n), total_seq / max(1, n), 1.0 - total_cer / max(1, n)


print("Helper functions defined ✔")


## 10 · Training Loop

Identical structure to the CTC baseline (`src/ctc/train.py` and
`train_colab_ctc_uncapped.ipynb`):
- Training dataset rebuilt each epoch so augmentation intensity follows the curriculum.
- Best checkpoint saved to Drive based on **validation sequence accuracy**.
- Gradient clipping at `CLIP_GRAD`.
- LR scheduler stepped **per batch** (warmup) then per epoch (cosine tail).
- Early stopping after `EARLY_STOP_PATIENCE` epochs without improvement.

In [ ]:
from tqdm import tqdm

# ── CSV metrics file ──────────────────────────────────────────────────────────
metrics_file = os.path.join(METRICS_DIR, 'transformer_metrics.csv')
if not os.path.exists(metrics_file):
    with open(metrics_file, 'w', newline='') as f:
        csv.writer(f).writerow([
            'epoch', 'train_loss', 'val_loss',
            'train_seq_acc', 'val_seq_acc',
            'train_char_acc', 'val_char_acc', 'lr',
        ])

LOG_EVERY          = 50   # compute train metrics every N steps
early_stop_counter = 0

# ─────────────────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    # Train dataset rebuilt each epoch so aug curriculum can update.
    # Reusing InfiniteCTCDataset from src/ctc/dataset.py — identical to CTC baseline.
    train_ds = InfiniteCTCDataset(
        digit_bank, config, size=TRAIN_SIZE, augment=True, epoch=epoch)
    train_loader = DataLoader(
        train_ds,
        batch_size   = BATCH_SIZE,
        collate_fn   = transformer_collate_fn,
        num_workers  = NUM_WORKERS,
        pin_memory   = True,
        persistent_workers = (NUM_WORKERS > 0),
    )

    # ── Train one epoch ───────────────────────────────────────────────────────
    model.train()
    total_loss = total_seq = total_cer = 0.0
    n_metric_steps = 0

    pbar = tqdm(enumerate(train_loader), total=STEPS_PER_EPOCH,
                desc="  train", leave=False)
    for step, batch in pbar:
        if step >= STEPS_PER_EPOCH:
            break

        images         = batch['images'].to(device)
        targets        = batch['targets'].to(device)
        target_lengths = batch['target_lengths'].to(device)

        optimizer.zero_grad()
        loss, logits = model(images, targets=targets, target_lengths=target_lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()

        # CRITICAL: step LR scheduler per batch during warmup.
        lr_scheduler.step()

        total_loss += loss.item()

        if (step + 1) % LOG_EVERY == 0 or (step + 1) == STEPS_PER_EPOCH:
            with torch.no_grad():
                # Fast teacher-forced accuracy (optimistic, for logging only).
                preds = logits.argmax(dim=-1)   # [B, L+1]
                decoded_fast = []
                for i, seq in enumerate(preds.tolist()):
                    L = int(target_lengths[i].item())
                    digits = [t for t in seq[:L] if t < 10]
                    decoded_fast.append(digits)
                seq_acc, cer = compute_metrics(decoded_fast, target_lengths, targets)
            total_seq      += seq_acc
            total_cer      += cer
            n_metric_steps += 1
            pbar.set_postfix(
                loss=f"{loss.item():.3f}",
                seq=f"{seq_acc:.3f}",
                cer=f"{cer:.3f}",
                lr=f"{lr_scheduler.current_lr:.2e}",
            )

    train_loss     = total_loss / STEPS_PER_EPOCH
    train_seq_acc  = total_seq  / max(1, n_metric_steps)
    train_char_acc = 1.0 - total_cer / max(1, n_metric_steps)

    # ── Validate (true autoregressive decoding) ────────────────────────────────
    val_loss, val_seq_acc, val_char_acc = validate(model, val_loader, device)

    current_lr = lr_scheduler.current_lr
    print(f"  Train | loss={train_loss:.4f}  seq_acc={train_seq_acc:.4f}  char_acc={train_char_acc:.4f}")
    print(f"  Val   | loss={val_loss:.4f}  seq_acc={val_seq_acc:.4f}  char_acc={val_char_acc:.4f}  lr={current_lr:.2e}")

    # ── Checkpoint ────────────────────────────────────────────────────────────
    if val_seq_acc > best_val_seq:
        best_val_seq       = val_seq_acc
        early_stop_counter = 0
        save_checkpoint(model, optimizer, epoch, val_loss, val_seq_acc, BEST_CKPT)
        print(f"  ✔ New best model saved (val_seq_acc={val_seq_acc:.4f})")
    else:
        early_stop_counter += 1

    # ── Log metrics to CSV ────────────────────────────────────────────────────
    with open(metrics_file, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch, train_loss, val_loss,
            train_seq_acc, val_seq_acc,
            train_char_acc, val_char_acc, current_lr,
        ])

    if early_stop_counter >= EARLY_STOP_PATIENCE:
        print(f"Early stopping triggered after {epoch} epochs.")
        break



## 11 · Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(metrics_file)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Transformer Seq2Seq Baseline — Training Curves', fontsize=13)

axes[0].plot(df['epoch'], df['train_loss'], label='train')
axes[0].plot(df['epoch'], df['val_loss'],   label='val')
axes[0].set_title('Cross-Entropy Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(df['epoch'], df['train_seq_acc'], label='train (teacher-forced)')
axes[1].plot(df['epoch'], df['val_seq_acc'],   label='val (autoregressive)')
axes[1].set_title('Sequence Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

axes[2].plot(df['epoch'], df['train_char_acc'], label='train')
axes[2].plot(df['epoch'], df['val_char_acc'],   label='val')
axes[2].set_title('Character Accuracy (1 − CER)')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.show()

## 12 · Length-Extrapolation Test (CRITICAL ablation result)

Load the best checkpoint and evaluate on sequences **longer than the training
maximum** (`max_length=12`). This demonstrates the **length prior failure mode**
of the global autoregressive Transformer.

Expected result (if the hypothesis holds):
- `L ≤ 12` (in-distribution): reasonable accuracy (~60–80%) ✅
- `L > 12` (OOD): catastrophic collapse — accuracy drops to <10% by L=50 ❌

This contrasts sharply with the CTC models which maintain ~88% digit accuracy
all the way to L=500, proving that non-autoregressive CTC is inherently
robust to length extrapolation.

In [ ]:
# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model.eval()
print(f"Loaded best checkpoint (epoch {ckpt['epoch']}, val_seq_acc={ckpt['val_seq_acc']:.4f})")

# ── Build a clean (no-aug) digit bank for deterministic evaluation ────────────
bank      = build_multidigit_bank(config['data_path'])
clean_aug = get_digit_aug_pipeline(augment=False, config=config)

# ── Evaluate at multiple sequence lengths ─────────────────────────────────────
N_SAMPLES = 200   # samples per length — higher → lower variance
EVAL_LENGTHS = [3, 5, 7, 10, 12, 15, 20, 30, 50, 100]
TRAIN_MAX    = 12   # anything above this is OOD

print(f"\n{'L':>4}  {'seq_acc':>8}  {'CER':>8}  {'status':>8}")
print("-" * 40)

results = []

for L in EVAL_LENGTHS:
    correct = 0
    total_ed, total_chars = 0, 0

    # Temporarily override config to force sequences of exactly length L.
    saved_min, saved_max = config['min_seq_len'], config['max_seq_len']
    config['min_seq_len'] = config['max_seq_len'] = L

    for _ in range(N_SAMPLES):
        img, true_digits = make_sequence(bank, clean_aug, config, augment=False, epoch=1)
        with torch.no_grad():
            # Inference mode: get encoder features then autoregressive decode.
            enc_feat = model(img.unsqueeze(0).to(device))   # [1, T, d_model]
            # Use a generous max_len cap (3x the sequence length).
            decoded  = model.greedy_autoregressive_decode(enc_feat, max_len=max(L * 3, 50))
        pred = decoded[0]

        if pred == true_digits:
            correct += 1
        total_ed    += _levenshtein(pred, true_digits)
        total_chars += L

    config['min_seq_len'] = saved_min
    config['max_seq_len'] = saved_max

    seq_acc = correct / N_SAMPLES
    cer     = total_ed / max(1, total_chars)
    ood_tag = "← OOD" if L > TRAIN_MAX else "in-dist"
    results.append({'L': L, 'seq_acc': seq_acc, 'cer': cer, 'ood': L > TRAIN_MAX})
    print(f"{L:>4}  {seq_acc:>8.3f}  {cer:>8.3f}  {ood_tag:>8}")

print("\nDone. Compare OOD rows against the CTC baseline — the collapse proves")
print("that absolute PE + autoregressive decoding induces a catastrophic length prior.")

## 13 · Inference on a Custom Image

In [ ]:
from google.colab import files
import torchvision.transforms as T
from PIL import Image

uploaded = files.upload()
if uploaded:
    img_path = list(uploaded.keys())[0]

    # Load and preprocess the image (same pipeline as the dataset).
    pil_img = Image.open(img_path).convert('L')   # grayscale
    transform = T.Compose([
        T.Resize((64, None)),   # fix height to 64, keep aspect ratio
        T.ToTensor(),
    ])
    img_tensor = transform(pil_img).unsqueeze(0).to(device)   # [1, 1, 64, W]

    model.eval()
    with torch.no_grad():
        enc_feat = model(img_tensor)   # inference mode
        decoded  = model.greedy_autoregressive_decode(enc_feat, max_len=200)

    predicted_digits = decoded[0]
    predicted_str    = ''.join(map(str, predicted_digits))

    print(f"Image           : {img_path}")
    print(f"Predicted digits: {predicted_digits}")
    print(f"Predicted string: '{predicted_str}'")
    print(f"Sequence length : {len(predicted_digits)}")